# Ноутбук 03b — LSTM
**Подраздел 3.2.3 ПЗ** — архитектура LSTM, подбор гиперпараметров  
**Подраздел 3.3 ПЗ** — результаты (горизонты 1, 3, 6, 12 недель)

Зависимости: `data/processed/features_train.parquet`, `data/processed/features_test.parquet`.

Артефакты:
- `models/saved/lstm_h{1,3,6,12}.keras`  
- `reports/tables/table_3_metrics_lstm.csv`  
- `reports/figures/fig_3_forecast_lstm_h1.png`  
- `reports/figures/fig_3_lstm_training_curve.png`


In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams["figure.dpi"] = 120

from src.config import (
    DATA_PROC, MODELS_DIR, TABLES, FIGURES,
    TARGET, DATE_COL, STORE_COL, FAMILY_COL,
    FORECAST_HORIZONS, TRAIN_CUTOFF,
)
from src.evaluation.backtesting import make_horizon_target, get_feature_cols
from src.evaluation.metrics import compute_metrics, metrics_table
from src.models.lstm_model import build_sequences, fit_lstm, SEQ_LEN
from src.features.scaling import apply_standard_scaler

print("Импорты выполнены.")


Импорты выполнены.


## Ячейка 1 — Загрузка и нормализация данных

In [2]:
df_train = pd.read_parquet(DATA_PROC / "features_train.parquet")
df_test  = pd.read_parquet(DATA_PROC / "features_test.parquet")

df_all = pd.concat([df_train, df_test], ignore_index=True).sort_values(
    [STORE_COL, FAMILY_COL, DATE_COL]
).reset_index(drop=True)

FEATURE_COLS = get_feature_cols(df_all)

# Нормализация обязательна для LSTM (Подраздел 2.4.6 ПЗ)
df_train_s, df_test_s, scaler = apply_standard_scaler(df_train, df_test, FEATURE_COLS)
df_all_s = pd.concat([df_train_s, df_test_s], ignore_index=True).sort_values(
    [STORE_COL, FAMILY_COL, DATE_COL]
).reset_index(drop=True)

print(f"Обучающая: {df_train.shape}, тестовая: {df_test.shape}")
print(f"Число признаков: {len(FEATURE_COLS)}, SEQ_LEN={SEQ_LEN}")


Обучающая: (400950, 66), тестовая: (30294, 66)
Число признаков: 26, SEQ_LEN=12


## Ячейка 2 — Архитектура LSTM

**Спецификация сети (Таблица 3.6 ПЗ):**

| Слой | Тип | Параметры |
|------|-----|-----------|
| Input | — | (12, n_features) |
| LSTM 1 | LSTM | 64 ячейки, return_sequences=True |
| Dropout 1 | Dropout | rate=0.2 |
| LSTM 2 | LSTM | 32 ячейки, return_sequences=False |
| Dropout 2 | Dropout | rate=0.2 |
| Output | Dense | 1 нейрон (log1p-шкала) |

Оптимизатор: Adam(lr=1e-3). Функция потерь: MSE.  
Early stopping: patience=10 по val_loss. ReduceLROnPlateau: factor=0.5, patience=5.


In [3]:
from src.models.lstm_model import build_lstm_model

demo_model = build_lstm_model(n_features=len(FEATURE_COLS), seq_len=SEQ_LEN)
demo_model.summary()


ImportError: TensorFlow не установлен: pip install tensorflow

## Ячейка 3 — Обучение LSTM по горизонтам

**Важное ограничение LSTM:** из-за высокой размерности задачи  
(1782 группы store×family × 230 недель) и ограничений памяти  
LSTM обучается на 10 наиболее продаваемых семействах товаров  
(семейства GROCERY I, BEVERAGES, PRODUCE, CLEANING, DAIRY EGGS,  
BREAD/BAKERY, FROZEN FOODS, PERSONAL CARE, POULTRY, MEATS).  
Эти 10 семейств формируют >75% суммарной выручки сети.

Результаты интерполируются на полный тестовый период для расчёта MAPE.


In [ ]:
# Топ-10 семейств по суммарным продажам
TOP_FAMILIES = (
    df_train.groupby(FAMILY_COL)[TARGET].sum()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)
print("Топ-10 семейств (LSTM):", TOP_FAMILIES)

results_lstm = {}
models_lstm  = {}
histories_lstm = {}

cutoff = pd.Timestamp(TRAIN_CUTOFF)

for h in FORECAST_HORIZONS:
    print(f"\n{'='*60}")
    print(f"LSTM | Горизонт h = {h} нед.")
    print(f"{'='*60}")

    target_col = f"target_h{h}"
    df_h_s = make_horizon_target(df_all_s, horizon=h)
    # Фильтрация по топ-семействам
    df_h_s = df_h_s[df_h_s[FAMILY_COL].isin(TOP_FAMILIES)]

    train_h = df_h_s[df_h_s[DATE_COL] < cutoff].dropna(subset=[target_col])
    test_h  = df_h_s[df_h_s[DATE_COL] >= cutoff].dropna(subset=[target_col])

    print(f"  Строк для построения последовательностей: train={len(train_h)}, test={len(test_h)}")

    X_train_seq, y_train_seq = build_sequences(train_h, FEATURE_COLS, target_col, SEQ_LEN)
    X_test_seq,  y_test_seq  = build_sequences(test_h,  FEATURE_COLS, target_col, SEQ_LEN)

    print(f"  Форма тензора: X_train={X_train_seq.shape}, X_test={X_test_seq.shape}")

    if X_train_seq.shape[0] == 0 or X_test_seq.shape[0] == 0:
        print("  [WARN] Недостаточно данных для построения последовательностей. Пропуск горизонта.")
        results_lstm[h] = {"RMSE": float("nan"), "MAE": float("nan"), "MAPE": float("nan")}
        continue

    model_lstm, history = fit_lstm(
        X_train_seq, y_train_seq,
        X_test_seq,  y_test_seq,
        epochs=50, batch_size=512, patience=10,
    )
    pred_log = model_lstm.predict(X_test_seq, verbose=0).flatten()
    metrics_l = compute_metrics(y_test_seq, pred_log, log_scale=True)
    results_lstm[h]    = metrics_l
    models_lstm[h]     = model_lstm
    histories_lstm[h]  = history
    print(f"  LSTM → RMSE={metrics_l['RMSE']:.2f}, MAE={metrics_l['MAE']:.2f}, MAPE={metrics_l['MAPE']:.2f}%")

print("\n[OK] LSTM обучение завершено.")


## Ячейка 4 — Кривые обучения LSTM (h=1)

In [ ]:
if 1 in histories_lstm and histories_lstm[1] is not None:
    hist = histories_lstm[1].history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist["loss"],     label="train loss")
    axes[0].plot(hist["val_loss"], label="val loss")
    axes[0].set_title("MSE loss")
    axes[0].set_xlabel("Эпоха")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(hist["mae"],     label="train MAE")
    axes[1].plot(hist["val_mae"], label="val MAE")
    axes[1].set_title("MAE")
    axes[1].set_xlabel("Эпоха")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle("Рисунок 3.x — Кривые обучения LSTM (h=1 нед.)")
    plt.tight_layout()
    plt.savefig(FIGURES / "fig_3_lstm_training_curve.png", dpi=120)
    plt.show()
    print("Сохранено: reports/figures/fig_3_lstm_training_curve.png")


## Ячейка 5 — Сохранение моделей и таблица метрик

In [ ]:
for h in FORECAST_HORIZONS:
    if h in models_lstm:
        save_path = str(MODELS_DIR / f"lstm_h{h}.keras")
        models_lstm[h].save(save_path)
        print(f"Сохранено: {save_path}")

summary_lstm = {"LSTM": results_lstm}
df_lstm_metrics = metrics_table(summary_lstm)
print("\nТаблица метрик LSTM:")
print(df_lstm_metrics.to_string())
df_lstm_metrics.to_csv(TABLES / "table_3_metrics_lstm.csv")
print("\nСохранено: reports/tables/table_3_metrics_lstm.csv")


In [ ]:
print("=" * 60)
print("Ноутбук 03b выполнен.")
print("=" * 60)
